In [ ]:
!pip install emlearn

In [ ]:
import numpy as np
import tensorflow as tf
from scipy import signal
import scipy.stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import json
import re
import traceback

try:
    import emlearn
except ImportError:
    emlearn = None
    print("Import Emlearn failed !")


print(tf.__version__)
print("TensorFlow Keras version:", tf.keras.__version__)

import scipy
print("SciPy version:", scipy.__version__)

In [ ]:
CONFIG = {
    'axes': 3,
    'scale_axes': 0.00010017430721,
    'filter_type': 'low',
    'filter_cutoff': 26.05078125,
    'filter_order': 6,
    'fft_length': 64,
    'do_fft_overlap': False,
    'do_log': True,
    'sampling_freq': 57,
    'raw_samples_per_axis': 57,
}

In [ ]:
FEATURES_PER_AXIS = 34  # 5 stats + 29 log10 FFT bins

def max_hold_psd(x, fs, fft_len, overlap=False):
    x = np.pad(x, (0, max(0, fft_len - len(x)))) if len(x) < fft_len else x
    n = len(x)
    num_bins = fft_len // 2 + 1
    psd_max = np.zeros(num_bins)
    stride = fft_len if not overlap else fft_len // 2
    window = np.hanning(fft_len)
    num_windows = (n - fft_len) // stride + 1
    for w in range(num_windows):
        start = w * stride
        seg = x[start:start + fft_len] * window
        fft_out = np.fft.rfft(seg)
        mag2 = (fft_out.real ** 2 + fft_out.imag ** 2) / (fft_len * np.sum(window ** 2))
        psd_max = np.maximum(psd_max, mag2)
    return psd_max


def extract_spec_features_axis(data_axis, config):
    fs = config['sampling_freq']
    fft_len = config['fft_length']
    x = data_axis.copy().astype(np.float64)

    # Butterworth lowpass filter
    if config['filter_type'] == 'low' and config['filter_order'] > 0:
        sos = signal.butter(config['filter_order'],
                            config['filter_cutoff'],
                            btype='low', fs=fs, output='sos')
        x = signal.sosfilt(sos, x)

    # Subtract mean
    x = x - np.mean(x)

    # Time-domain stats
    n = len(x)
    rms = float(np.sqrt(np.mean(x ** 2)))
    std = np.std(x, ddof=0)
    if std == 0:
        std = 1e-10
    skewness = float(np.mean(x ** 3) / (std ** 3))
    kurtosis = float(np.mean(x ** 4) / (std ** 4) - 3)

    # Welch max-hold spectrum
    psd = max_hold_psd(x, fs, fft_len, overlap=config['do_fft_overlap'])

    # Skewness and kurtosis of PSD values (all PSD bins)
    psd_safe = np.maximum(psd, 1e-10)
    psd_skew = scipy.stats.skew(psd_safe)
    psd_skew = float(psd_skew) if np.isfinite(psd_skew) else 0.0
    psd_kurt = scipy.stats.kurtosis(psd_safe)
    psd_kurt = float(psd_kurt) if np.isfinite(psd_kurt) else 0.0

    # Log10 of bins [1..29] (skip DC, up to filter cutoff)
    cutoff_hz = config['filter_cutoff']
    bin_resolution = fs / fft_len
    stop_bin = int(np.floor(cutoff_hz / bin_resolution + 0.5)) + 1
    stop_bin = min(stop_bin, len(psd))
    start_bin = 1
    num_bins_used = stop_bin - start_bin  # should be 29

    log_bins = np.zeros(num_bins_used)
    for i in range(num_bins_used):
        val = max(psd[start_bin + i], 1e-10)
        log_bins[i] = float(np.log10(val))

    features = [rms, skewness, kurtosis, psd_skew, psd_kurt] + log_bins.tolist()
    return features


def extract_features(raw_window, config=CONFIG):
    x = raw_window * config['scale_axes']
    features = []
    for axis in range(config['axes']):
        feat = extract_spec_features_axis(x[:, axis], config)
        features.extend(feat)
    return np.array(features, dtype=np.float32)


# ============================================================
# DATA AUGMENTATION
# ============================================================

def augment_gaussian_noise(raw_window, std_range=(0.01, 0.05)):
    """Add Gaussian noise to raw signal."""
    std = np.random.uniform(*std_range)
    noise = np.random.normal(0, std, raw_window.shape)
    return raw_window + noise


def augment_time_shift(raw_window, max_shift=5):
    """Shift signal in time."""
    shift = np.random.randint(-max_shift, max_shift + 1)
    return np.roll(raw_window, shift, axis=0)


def augment_amplitude_scale(raw_window, scale_range=(0.8, 1.2)):
    """Scale amplitude."""
    scale = np.random.uniform(*scale_range)
    return raw_window * scale


def augment_axis_swap(raw_window):
    """Randomly swap two axes."""
    w = raw_window.copy()
    axes_pairs = [(0, 1), (1, 2), (0, 2)]
    a, b = axes_pairs[np.random.randint(0, 3)]
    w[:, [a, b]] = w[:, [b, a]]
    return w


def augment_sample(raw_window, config=CONFIG):
    """Apply random combination of augmentations."""
    w = raw_window.copy()
    # Always add noise
    w = augment_gaussian_noise(w)
    # Randomly apply 1-2 more augmentations
    augmentations = [augment_time_shift, augment_amplitude_scale, augment_axis_swap]
    np.random.shuffle(augmentations)
    n_extra = np.random.randint(1, 3)
    for aug_fn in augmentations[:n_extra]:
        w = aug_fn(w)
    return w


def augment_dataset(raw_data, labels, num_augments=5, config=CONFIG):
    """Augment entire dataset. Returns original + augmented samples."""
    X_aug = []
    y_aug = []
    for i in range(len(raw_data)):
        # Original
        feat = extract_features(raw_data[i], config)
        X_aug.append(feat)
        y_aug.append(labels[i])
        # Augmented copies
        for _ in range(num_augments):
            aug_raw = augment_sample(raw_data[i])
            feat = extract_features(aug_raw, config)
            X_aug.append(feat)
            y_aug.append(labels[i])
    return np.array(X_aug, dtype=np.float32), np.array(y_aug, dtype=np.int32)



2. DATASET PREPARATION 

In [ ]:
def prepare_dataset(raw_data, labels, config=CONFIG):
    X = []
    for i in range(len(raw_data)):
        feat = extract_features(raw_data[i], config)
        X.append(feat)
    return np.array(X, dtype=np.float32), np.array(labels)


def load_json_data(dataset_path, config):
    all_raw_windows = []
    all_labels = []

    label_map = {}
    # Use training/info.labels to avoid extra labels from testing/root
    labels_filepath = os.path.join(dataset_path, 'training', 'info.labels')
    if not os.path.exists(labels_filepath):
        labels_filepath = os.path.join(dataset_path, 'info.labels')
    if not os.path.exists(labels_filepath):
        raise FileNotFoundError(f"info.labels not found at {labels_filepath}")
    with open(labels_filepath, 'r') as f:
        labels_info = json.load(f)

    unique_labels = sorted(list(set(item['label']['label'] for item in labels_info['files'] if 'label' in item and 'label' in item['label'])))
    unique_labels = [l for l in unique_labels if l != 'testing']
    for i, label_name in enumerate(unique_labels):
        label_map[label_name] = i

    if not unique_labels:
        raise ValueError("No unique labels found in info.labels. Check the JSON structure.")

    print(f"Label mapping: {label_map}")

    window_size = config['raw_samples_per_axis']
    stride = window_size // 2 

    subdirs = ['training']
    for subdir in subdirs:
        current_dir = os.path.join(dataset_path, subdir)
        if not os.path.isdir(current_dir):
            print(f"Warning: Directory '{current_dir}' not found. Skipping.")
            continue

        for filename in os.listdir(current_dir):
            if filename.endswith('.json'):
                filepath = os.path.join(current_dir, filename)
                try:
                    with open(filepath, 'r') as f:
                        data = json.load(f)

                    raw_recording_values = data['payload']['values']
                    raw_recording = np.array(raw_recording_values, dtype=np.float32)

                    # Get label from JSON directly, accessing nested 'label' field
                    label_name = data.get('label', {}).get('label')
                    if label_name is None:
                        # Fallback: Extract label from filename
                        match = re.match(r'([a-zA-Z0-9_\-]+)\.json', filename)
                        if match:
                            label_name = match.group(1)

                    if label_name not in label_map:
                        print(f"Warning: Label '{label_name}' from file '{filename}' not found in info.labels or could not be extracted. Skipping entire recording.")
                        continue

                    current_recording_length = raw_recording.shape[0]
                    if current_recording_length < window_size:
                        print(f"Warning: Recording in '{filename}' has length {current_recording_length}, which is less than expected window size {window_size}. Skipping.")
                        continue

                    # Extract windows from the raw recording
                    num_windows_in_recording = (current_recording_length - window_size) // stride + 1

                    for i in range(num_windows_in_recording):
                        start_idx = i * stride
                        end_idx = start_idx + window_size

                        # Ensure the window is exactly the expected size (116, 3)
                        window = raw_recording[start_idx:end_idx]
                        if window.shape == (window_size, config['axes']):
                            all_raw_windows.append(window)
                            all_labels.append(label_map[label_name])
                        else:
                            print(f"Warning: Extracted window from '{filename}' has shape {window.shape}, expected {(window_size, config['axes'])}. Skipping this window.")

                except KeyError as ke:
                    print(f"Error: Missing key in JSON file {filepath} (e.g., 'payload' or 'label'): {ke}. Skipping.")
                except Exception as e:
                    print(f"Error processing file {filepath}: {e}. Skipping.")

    print(f"Loaded {len(all_raw_windows)} valid windows after parsing and windowing.")
    if not all_raw_windows:
        raise ValueError("No data loaded from JSON files after windowing. Check paths, JSON structure, info.labels, and windowing logic.")

    # Per-class count
    labels_arr = np.array(all_labels, dtype=np.int32)
    print("\n--- Per-class window count ---")
    for cls_name in sorted(label_map.keys(), key=lambda x: label_map[x]):
        count = (labels_arr == label_map[cls_name]).sum()
        print(f"  {cls_name:<10} (id={label_map[cls_name]}): {count} windows")
    print(f"  {'TOTAL':<10}: {len(labels_arr)} windows")

    return np.array(all_raw_windows, dtype=np.float32), labels_arr

3. MODEL DEFINITION
Architecture (matching Edge Impulse v4 spectral analysis)
    Input:  (1, 102) int8   -- 34 features x 3 axes
    FC1:    20 units, ReLU   weights (20x102)
    FC2:    10 units, ReLU   weights (10x20)
    FC3:    6 units, Linear  weights (6x10)
    Softmax

Feature layout per axis (34 features):
    [0]  RMS, [1] Skewness, [2] Kurtosis
    [3]  Spectrum Skewness, [4] Spectrum Kurtosis
    [5..33] 29 log10 FFT bins (Welch max-hold, 64-pt FFT, no overlap)

In [ ]:
def build_model(input_dim, num_classes=6):
    from tensorflow.keras import regularizers
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(20, activation='relu', kernel_regularizer=regularizers.l2(0.01), name='fc1'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation='relu', kernel_regularizer=regularizers.l2(0.01), name='fc2'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax', name='fc3'),
    ])
    return model

def train(X_train, y_train, X_val=None, y_val=None, epochs=50, num_classes=6, class_weight=None):
    model = build_model(input_dim=X_train.shape[1], num_classes=num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    ]

    val_data = (X_val, y_val) if X_val is not None else None
    history = model.fit(
        X_train, y_train,
        validation_data=val_data,
        epochs=epochs,
        batch_size=16,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )
    return model, history


def fit_scaler(X_train):
    with np.errstate(divide='ignore', invalid='ignore'):
        scaler = StandardScaler(with_mean=True, with_std=True)
        scaler.fit(X_train)
    scaler.scale_[np.isinf(scaler.scale_) | np.isnan(scaler.scale_) | (scaler.scale_ == 0)] = 1.0
    return scaler


def representative_dataset(X_calib):
    for i in range(len(X_calib)):
        feat = X_calib[i]
        yield [feat.reshape(1, -1).astype(np.float32)]


def export_tflite(model, scaler, X_calib, config=CONFIG,
                  filename='model_quantized.tflite', input_dim=102):
    feature_input = tf.keras.layers.Input(shape=(input_dim,), name='feature_input')

    # StandardScaler uses (x - mean) / scale.
    mean = tf.constant(scaler.mean_, dtype=tf.float32)
    scale = tf.constant(scaler.scale_, dtype=tf.float32)
    norm_layer = tf.keras.layers.Lambda(
        lambda x: (x - mean) / scale,
        name='standard_scaler'
    )(feature_input)

    output = model(norm_layer)
    full_model = tf.keras.Model(inputs=feature_input, outputs=output)

    converter = tf.lite.TFLiteConverter.from_keras_model(full_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    def rep_gen():
        for i in range(min(200, len(X_calib))):
            feat = X_calib[i]
            yield [feat.reshape(1, -1).astype(np.float32)]
    converter.representative_dataset = rep_gen

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(filename, 'wb') as f:
        f.write(tflite_model)
    print(f"TFLite model saved: {filename} ({len(tflite_model)} bytes)")

    return tflite_model


# ============================================================
# 8. MAIN
# ============================================================

if __name__ == '__main__':
    print("=== Motion Direct Classify Training Pipeline ===")
    print("\nLoading dataset: motion-direct-classify-export from JSON files..." + os.getcwd())
    dataset_path = os.path.join(os.getcwd(), 'motion-direct-classify-export')

    try:
        # Load raw data (before feature extraction)
        raw_data, labels = load_json_data(dataset_path, CONFIG)
        print(f"Raw data shape: {raw_data.shape}")
        print(f"Labels shape: {labels.shape}")

        # Augment dataset: original + 5 augmented copies per sample
        print("\nAugmenting dataset (5x)...")
        X, y = augment_dataset(raw_data, labels, num_augments=5, config=CONFIG)
        print(f"After augmentation: X={X.shape}, y={y.shape}")

        num_classes = len(np.unique(y))

        # K-Fold Cross Validation
        from sklearn.model_selection import StratifiedKFold
        from sklearn.utils.class_weight import compute_class_weight
        kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        fold_scores = []
        best_val_acc = 0
        best_model = None
        best_scaler = None

        print(f"\nStarting 5-Fold Cross Validation...")
        for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
            print(f"\n{'='*50}")
            print(f"Fold {fold + 1}/5")
            print(f"{'='*50}")

            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]

            # Class distribution
            print("Train:", {c: int((y_tr == c).sum()) for c in range(num_classes)})
            print("Val:  ", {c: int((y_val == c).sum()) for c in range(num_classes)})

            # Fit scaler on train only
            scaler = fit_scaler(X_tr)
            X_tr_norm = scaler.transform(X_tr)
            X_val_norm = scaler.transform(X_val)

            # Class weights
            cw = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_tr)
            class_weight_dict = {i: w for i, w in enumerate(cw)}

            # Train
            model, history = train(X_tr_norm, y_tr, X_val_norm, y_val,
                                   epochs=50, num_classes=num_classes,
                                   class_weight=class_weight_dict)

            # Evaluate
            val_loss, val_acc = model.evaluate(X_val_norm, y_val, verbose=0)
            fold_scores.append({'loss': val_loss, 'accuracy': val_acc})
            print(f"Fold {fold+1} result: val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = model
                best_scaler = scaler
                print(f"  -> New best model (acc={val_acc:.4f})")

        # Summary
        print(f"\n{'='*50}")
        print("K-Fold Results:")
        avg_loss = np.mean([s['loss'] for s in fold_scores])
        avg_acc = np.mean([s['accuracy'] for s in fold_scores])
        std_acc = np.std([s['accuracy'] for s in fold_scores])
        print(f"  Avg val_loss: {avg_loss:.4f}")
        print(f"  Avg val_acc:  {avg_acc:.4f} +/- {std_acc:.4f}")
        print(f"  Best val_acc: {best_val_acc:.4f}")

        # Export best model
        print("\nExporting TFLite model (best fold)...")
        tflite_model = export_tflite(best_model, best_scaler, X,
                                      filename='motion_direct_classify_model_quantized.tflite',
                                      input_dim=X.shape[1])

    except Exception as e:
        print(f"Error during data loading or processing: {e}")
        print("Skipping training and subsequent steps.")
        traceback.print_exc()

In [ ]:
def export_emlearn(model, filename='motion_direct_classify_model.h', name='motion_direct_model'):
    if emlearn is None:
        raise ImportError(
            "emlearn is not installed. Install it with `pip install emlearn`."
        )

    cmodel = emlearn.convert(model, method='inline')
    cmodel.save(file=filename, name=name)
    print(f"emlearn model saved: {filename}")
    return cmodel

print("\nExporting C/C++ model via emlearn...")
emlearn_model = export_emlearn(model, filename='motion_direct_classify_model.h', name='motion_direct_model')

print("\n/* Copy these scaler values to C++ */")
print("static const float NORM_MEAN[FEATURE_LEN] = {", ", ".join(f"{v:.4f}f" for v in scaler.mean_), "};")
print("static const float NORM_SCALE[FEATURE_LEN] = {", ", ".join(f"{v:.6f}f" for v in 1.0 / scaler.scale_), "};")
print(f"\nNum classes: {num_classes}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_val_norm)
y_pred_classes = np.argmax(y_pred, axis=1)

target_names = ['down', 'idle', 'left', 'right', 'unknown', 'up']

print('Classification Report (Validation Set):')
print(classification_report(y_val, y_pred_classes, labels=list(range(num_classes)), target_names=target_names, zero_division=0))

cm = confusion_matrix(y_val, y_pred_classes, labels=list(range(num_classes)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Validation')
plt.savefig('confusion_matrix.png')
plt.show()

In [ ]:
# ============================================================
# CONFIDENCE THRESHOLD ANALYSIS
# Run model on ALL data, analyze per-class softmax outputs
# ============================================================

labels_map = {0: 'down', 1: 'idle', 2: 'left', 3: 'right', 4: 'unknown', 5: 'up'}

# Normalize ALL data
X_all_norm = scaler.transform(X)

# Get softmax outputs for all samples
y_prob_all = model.predict(X_all_norm)

print("="*70)
print("CONFIDENCE THRESHOLD ANALYSIS")
print("="*70)
print(f"{'Class':<10} {'Count':<6} {'Min':<8} {'Mean':<8} {'Std':<8} {'Suggested':<10}")
print("-"*70)

suggested_conf = {}
for c in range(num_classes):
    mask = (y == c)
    count = mask.sum()
    if count == 0:
        print(f"{labels_map[c]:<10} {count:<6} (no samples)")
        suggested_conf[c] = 0.0
        continue
    probs_of_correct_class = y_prob_all[mask, c]
    min_prob = probs_of_correct_class.min()
    mean_prob = probs_of_correct_class.mean()
    std_prob = probs_of_correct_class.std()
    # Threshold = min - 1 std (but not below 0)
    threshold = max(0.0, min_prob - 1.0 * std_prob)
    suggested_conf[c] = threshold
    print(f"{labels_map[c]:<10} {count:<6} {min_prob:<8.4f} {mean_prob:<8.4f} {std_prob:<8.4f} {threshold:<10.4f}")

print("="*70)
print("\nSuggested confidence thresholds (for app.cpp):")
print("  MotionDirectConfidence_t conf = {0.0f};")
for c in range(num_classes):
    print(f"  conf.{labels_map[c]} = {suggested_conf[c]:.4f}f;")

# Also show: what if we use mean - 2*std?
print("\n--- Alternative: mean - 2*std (more conservative) ---")
for c in range(num_classes):
    mask = (y == c)
    if mask.sum() == 0:
        continue
    probs = y_prob_all[mask, c]
    threshold = max(0.0, probs.mean() - 2.0 * probs.std())
    print(f"  conf.{labels_map[c]} = {threshold:.4f}f;")